In [1]:
"""
Basic
"""
import sys
sys.path.insert( 0, "C:/users/hans/OneDrive/Python3/packages/cdxcore")

import importlib as imp
import cdxcore.deferred as _
imp.reload(_)

from cdxcore.deferred import Deferred
from cdxcore.verbose import Context

class A(object):
    def __init__(self):
        self.x = 1
    def f(self,other):
        return self.x+(other.x if not isinstance(other, int) else other)
        
a = Deferred("A")
a.f(1)

a.deferred_print_dependency_tree()

print("\nResolving...")
a.deferred_resolve( A(), Context.all )


00: $A
01:   $A.f
02:     $A.f(1)

Resolving...
00: $A -> 'A' : <__main__.A object at 0x7d7fced54cb0>
01:   $A.f -> 'A.f' : <bound method A.f of <__main__.A object at 0x7d7fced54cb0>>
02:     $A.f(1) -> 'int' : 2


In [2]:
"""
Test all operators and some standard python features 
"""

A = None


import sys
sys.path.insert( 0, "C:/users/hans/OneDrive/Python3/packages/cdxcore")

import importlib as imp
import cdxcore.deferred as _
imp.reload(_)

from cdxcore.deferred import DeferAll
from cdxcore.verbose import Context

class qB(object):
    def __init__(self):
        self.m = 1
    def f(self,y):
        return self.m+y

class qA(object):

    M = 7

    def __init__(self):
        self.m = 1
        self.d = dict()
        self.l = list()
        self.b = qB()
    
    def f(self, y=2):
        return self.m*y

    def fq(self, q):
        return self.m*q.m
    
    @property
    def g(self):
        return self.m*3

    @staticmethod
    def h(x,*,y=2):
        return x*y

    @property
    def more(self):
        return qB()

    def moref(self):
        return qB()

    @classmethod
    def j(cls, y=2):
        return cls.M*y
        
    def __iter__(self):
        for i in  range(self.x):
            yield i
            
    def __call__(self, x):
        return self.m*x
    
    def __getitem__(self, i):
        return self.d[i]
    
    def __setitem__(self, i, v):
        self.d[i] = v

    def __eq__(self, other):
        return self.m == other.m

    def __iadd__(self, integer):
        self.m += integer
        return self

    def __rxor__(self, integer):
        r = qA()
        r.m = self.m^integer
        return r

def some_function(x):
    return x.m

b = DeferAll("b")
print("Launch")

a = DeferAll("a")

def test_iadd(a):
    a+=1
    return a

def tester(a,b):
    am = a.m
    return dict(
        aM = a.M,
        am = am,
        ag = a.g,
        af2 = a.f(2),
        afm = a.f(am),
        af23 = a.f(y=2)*3+1,
        ah = a.h(x=3,y=4),
        aj = a.j(5),
        a2j = 2+a.j(5),
        amm = a.more.m,
        amff = a.moref().f(y=3),
        afqb = a.fq( b ),
        eq = a==b,
        iadd = test_iadd(a).m,
        rxor = (1^a).m,#
        some = some_function(a)
    )

deferred_a = DeferAll("a")
deferred_b = DeferAll("b")
actual_a = qA()
actual_b = qA()
results_act = tester(actual_a, actual_b)
results_drf = tester(deferred_a, deferred_b)

print(results_act)

deferred_b.deferred_resolve( qA() )
deferred_a.deferred_resolve( qA(), verbose=Context.all )

print("Did it work?")
results_tst = { k: v.deferred_result for k,v in results_drf.items() }

print(results_tst)

# deferred as arguments
print("Manual tree")

def generate(x,i=0):
    s = f"'{i}: {x.deferred_info} <- "
    for src in x.deferred_sources_names:
        s += src + ","
    s = s[:-1] + "'\n"
    for d in x.deferred_dependants:
        s += generate(d,i+1)
    return s
print( generate(deferred_a) )


print("Dependency Tree")

collect = []
a.deferred_print_dependency_tree(verbose=Context("all", channel=lambda message, flush : collect.append( message ) ))
print(collect)

print("\nResolve")
b.deferred_resolve( qA() )
a.deferred_resolve( qA() )
print("resolved:", deferred_a.deferred_result )


Launch
{'aM': 7, 'am': 1, 'ag': 3, 'af2': 2, 'afm': 1, 'af23': 7, 'ah': 12, 'aj': 35, 'a2j': 37, 'amm': 1, 'amff': 4, 'afqb': 1, 'eq': True, 'iadd': 2, 'rxor': 3, 'some': 2}
00: $a -> 'qA' : <__main__.qA object at 0x7d7fce28f7d0>
01:   $a.m -> 'int' : 1
01:   $a.M -> 'int' : 7
01:   $a.g -> 'int' : 3
01:   $a.f -> 'qA.f' : <bound method qA.f of <__main__.qA object at 0x7d7fce28f7d0>>
02:     $a.f(2) -> 'int' : 2
01:   $a.f -> 'qA.f' : <bound method qA.f of <__main__.qA object at 0x7d7fce28f7d0>>
02:     $a.f({$a.m}) -> 'int' : 1
01:   $a.f -> 'qA.f' : <bound method qA.f of <__main__.qA object at 0x7d7fce28f7d0>>
02:     $a.f(y=2) -> 'int' : 2
03:       ($a.f(y=2)*3) -> 'int' : 6
04:         (($a.f(y=2)*3)+1) -> 'int' : 7
01:   $a.h -> 'qA.h' : <function qA.h at 0x7d7fcdc980e0>
02:     $a.h(x=3,y=4) -> 'int' : 12
01:   $a.j -> 'qA.j' : <bound method qA.j of <class '__main__.qA'>>
02:     $a.j(5) -> 'int' : 35
01:   $a.j -> 'qA.j' : <bound method qA.j of <class '__main__.qA'>>
02:     $a

In [3]:
import numpy as np

def c1(a):
	a+=3
	return a
def c2(a):
	a-=3
	return a
def c3(a):
    a = a.astype(np.float32)
    a/=3
    return a
def c4(a):
	a//=3
	return a
def c5(a):
	a*=3
	return a
def c6(a):
	a^=3
	return a
def c7(a):
	a|=3
	return a
def c8(a):
	a&=3
	return a
def c9(a):
	a**=3
	return a
def cA(a):
    a@=a.T
    return a
def cB(a):
	a%=3
	return a
def test_op(a):
    return dict(
		# left
        a1 = a+3    ,
        a2 = a-3    ,
        a3 = a/3    ,
        a4 = a//3   ,
        a5 = a*3    ,
        a6 = a^3    ,
        a7 = a|3    ,
        a8 = a&3    ,
        a9 = a**3   ,
        aA = a@np.full((2,1),2)    ,
        aB = a%3    ,
		# right
        b1 = 3+a    ,
        b2 = 3-a    ,
        b3 = 3/a    ,
        b4 = 3//a   ,
        b5 = 3*a    ,
        b6 = 3^a    ,
        b7 = 3|a    ,
        b8 = 3&a    ,
        b9 = 3**a   ,
        bA = a.T @ a,
        bB = 3%a    ,
        bC = [3]+a  ,
		# in place
		c1 = c1(a),
		c2 = c2(a),
        c3 = c3(a),
        c4 = c4(a),
        c5 = c5(a),
        c6 = c6(a),
        c7 = c7(a),
        c8 = c8(a),
        c9 = c9(a),
        cB = cB(a)
	)



deferred_a = DeferAll("a")
actual_a = np.full((4,2),3,dtype=np.int32)
results_act = test_op(actual_a)
results_drf = test_op(deferred_a)

print(results_act)

deferred_a.deferred_resolve( np.full((4,2),3,dtype=np.int32) )

print("Did it work?")
results_act = { k: v.astype(np.int32) for k,v in results_act.items() }
results_tst = { k: v.deferred_result.astype(np.int32) for k,v in results_drf.items() }

print(results_tst)

print("/done")

{'a1': array([[6, 6],
       [6, 6],
       [6, 6],
       [6, 6]], dtype=int32), 'a2': array([[0, 0],
       [0, 0],
       [0, 0],
       [0, 0]], dtype=int32), 'a3': array([[1., 1.],
       [1., 1.],
       [1., 1.],
       [1., 1.]]), 'a4': array([[1, 1],
       [1, 1],
       [1, 1],
       [1, 1]], dtype=int32), 'a5': array([[9, 9],
       [9, 9],
       [9, 9],
       [9, 9]], dtype=int32), 'a6': array([[0, 0],
       [0, 0],
       [0, 0],
       [0, 0]], dtype=int32), 'a7': array([[3, 3],
       [3, 3],
       [3, 3],
       [3, 3]], dtype=int32), 'a8': array([[3, 3],
       [3, 3],
       [3, 3],
       [3, 3]], dtype=int32), 'a9': array([[27, 27],
       [27, 27],
       [27, 27],
       [27, 27]], dtype=int32), 'aA': array([[12],
       [12],
       [12],
       [12]]), 'aB': array([[0, 0],
       [0, 0],
       [0, 0],
       [0, 0]], dtype=int32), 'b1': array([[6, 6],
       [6, 6],
       [6, 6],
       [6, 6]], dtype=int32), 'b2': array([[0, 0],
       [0, 0],
       [0

In [4]:
"""
Test all operators and some standard python features 
"""

A = None


import sys
sys.path.insert( 0, "C:/users/hans/OneDrive/Python3/packages/cdxcore")

import importlib as imp
import cdxcore.deferred as _
imp.reload(_)

from cdxcore.deferred import DeferAll
from cdxcore.verbose import Context

class A(object):
    def __init__(self, x):
        self.x = x
    def f(self, y=1):
        return self.x * y
    def __call__(self, y):
        return self.x + y
    def __eq__(self, z):
        return self.x==z

def F(a : A):       
    _ = a.f(2)
    _ = _+a(3)  
    return _, _ == 1

a      = DeferAll("a")
r1, r2 = F(a)
a.deferred_print_dependency_tree( with_sources = True )
a.deferred_resolve(A(x=2))
print("Deferred results:", r1.deferred_result, ",", r2.deferred_result)

t1, t2 = F(A(x=2))
print("Deferred results:", t1, ",", t2)


00: $a <= $a
01:   $a.f <= $a
02:     $a.f(2) <= $a
03:       ($a.f(2)+$a(3)) <= $a
04:         (($a.f(2)+$a(3))!=1) <= $a
01:   $a(3) <= $a
Deferred results: 9 , False
Deferred results: 9 , False
